# Extracting whole genome genotype data 

This is a script to extract genotypes from whole genomes of *An.gambiae* collected during the LLINEUP trial that was conducted in Uganda from 2017-2019.  



In [1]:
#Install and load packages
import malariagen_data
import os
import numpy as np
import pandas as pd
import allel
import xarray as xr
import glob

/home/harunnn/.conda/envs/gaard/lib/python3.10/site-packages/anjl/_canonical.py:187: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  n_threads = get_num_threads()


In [ ]:

ag3 = malariagen_data.Ag3(pre = True)

In [ ]:

#function to extract biallelic genotypes

def extract_and_filter_snps(region, maf_threshold,output_filename):
    
    array_snps = ag3.snp_calls(region=region,
                               sample_sets=["1288-VO-UG-DONNELLY-VMF00168","1288-VO-UG-DONNELLY-VMF00219"],
                               sample_query=("aim_species == 'gambiae'"),
                               site_mask='gamb_colu' )
    
    gt = allel.GenotypeArray(array_snps['call_genotype'])
    
    no_missing = gt.count_missing(1) == 0
    gt_freq=gt.count_alleles().to_frequencies()
    which_pos = (np.max(gt_freq,1) < (1 - maf_threshold)) & no_missing
    gt_filtered = gt[which_pos,:]
   
    gt_biallelic = np.sum(gt_filtered>0,2)
    #convert to dataframe
    df_gt = pd.DataFrame(gt_biallelic)
    pos = array_snps['variant_position'][which_pos]
    
    chrom = np.array(array_snps.contigs)[array_snps['variant_contig']][which_pos]
    #snp_id = np.apply_along_axis(':'.join, 0, [chrom, pos.astype('str')])
    snp_id = np.apply_along_axis(lambda x: np.asarray(':'.join(x), dtype = 'object'), 0, [chrom, pos.values.astype('str')])
    df_gt.set_index(snp_id, inplace = True)
    df_gt.columns=array_snps.sample_id
    # Save DataFrame to CSV file
    df_gt.to_csv(output_filename)
    return(df_gt)
    



In [ ]:
output_directory = '/llineup/llineup-genomics/data/glm_genotypes'

regions = ['2L', '2R', '3L', '3R', 'X']

# Dictionary comprehension to call the function for each region
gt = {region: extract_and_filter_snps(region, 0.02, os.path.join(output_directory, f'gt_{region}.csv')) for region in regions}


In [ ]:
#Save genotypes

path = '/llineup/llineup-genomics/data/glm_genotypes'
all_csv = glob.glob(path + "**/*gt_*.csv")
df_list = [pd.read_csv(filename, index_col=None) for filename in all_csv]
df_gt = pd.concat(df_list,axis=0, ignore_index=True)
df_gt.to_csv(path+ "/gt_glm.csv")

In [ ]:
#extract genotype for fst populations differentiation analysis
#load malariagen data
ag3 = malariagen_data.Ag3(pre = True)
meta = ag3.sample_metadata(
    sample_sets=["1288-VO-UG-DONNELLY-VMF00168","1288-VO-UG-DONNELLY-VMF00219"], 
    sample_query = "aim_species == 'gambiae'"
)
# Remove decimal and numbers after it in the "partner_sample_id" column not present in our meta data
meta['partner_sample_id'] = meta['partner_sample_id'].str.split('.').str[0]

# Convert "partner_sample_id" column in meta to float64 to match llineup_meta dtype
meta['partner_sample_id'] = pd.to_numeric(meta['partner_sample_id'])
#llineup trial metadata
llineup_meta = pd.read_csv('~/llineup/llineup-genomics/data/ento_geno_plasmo_data.csv',
                       index_col = 0,
                      ).query("species=='An. gambiae'")
llineup_meta= llineup_meta.drop_duplicates(subset=['wgs.sample.id'])
llineup_meta= llineup_meta.set_index('wgs.sample.id')
llineup_meta = llineup_meta.reindex(index=meta['partner_sample_id'])#match order in malariagen meta data
llineup_meta=llineup_meta.reset_index()
llineup_meta['sample_id'] =meta['sample_id'] #create sample id to merge with ag3 metadata

#add column control phase for grouping samples to pre and post interventions
def recode_rnd(rnd):
    if rnd == 1:
        return 'pre'
    elif rnd == 5:
        return 'post'
    else:
        return 'intermediate'

llineup_meta['control_phase'] = llineup_meta['RND'].apply(recode_rnd)
llineup_meta.rename(columns={'LLIN.actual':'llin_actual'}, inplace = True)
llineup_meta.drop(columns=['partner_sample_id'], inplace=True)
ag3.add_extra_metadata(llineup_meta)


#Modify function to extract biallelic genotypes for fst analysis

def extract_and_filter_snps(region, maf_threshold,output_filename):
    
    array_snps = ag3.snp_calls(region=region,
                               sample_sets=["1288-VO-UG-DONNELLY-VMF00168","1288-VO-UG-DONNELLY-VMF00219"],
                               sample_query=("aim_species == 'gambiae' and control_phase == 'pre'"),
                               site_mask='gamb_colu' )
    
    gt = allel.GenotypeArray(array_snps['call_genotype'])
    
    no_missing = gt.count_missing(1) == 0
    gt_freq=gt.count_alleles().to_frequencies()
    which_pos = (np.max(gt_freq,1) < (1 - maf_threshold)) & no_missing
    gt_filtered = gt[which_pos,:]
   
    gt_biallelic = np.sum(gt_filtered>0,2)
    #convert to dataframe
    df_gt = pd.DataFrame(gt_biallelic)
    pos = array_snps['variant_position'][which_pos]
    
    chrom = np.array(array_snps.contigs)[array_snps['variant_contig']][which_pos]
    #snp_id = np.apply_along_axis(':'.join, 0, [chrom, pos.astype('str')])
    snp_id = np.apply_along_axis(lambda x: np.asarray(':'.join(x), dtype = 'object'), 0, [chrom, pos.values.astype('str')])
    df_gt.set_index(snp_id, inplace = True)
    df_gt.columns=array_snps.sample_id
    # Save DataFrame to CSV file
    df_gt.to_csv(output_filename)
    return(df_gt)
    




In [ ]:

output_directory = '/llineup/llineup-genomics/data/fst_genotypes'

regions = ['2', '2R', '3L', '3R', 'X']

# Dictionary comprehension to call the function for each region
gt_pre = {region: extract_and_filter_snps(region, 0.02, os.path.join(output_directory, f'gt_pre_{region}.csv')) for region in regions}


In [ ]:
#Save genotypes

path = '/llineup/llineup-genomics/data/fst_genotypes'
all_csv = glob.glob(path + "**/*gt_pre_*.csv")
df_list = [pd.read_csv(filename, index_col=None) for filename in all_csv]
df_gt = pd.concat(df_list,axis=0, ignore_index=True)
df_gt.to_csv(path+ "/gt_pre_fst.csv")